# DeltaL analysis for topologies A1–A4

This notebook contains only the analyses reported in the manuscript and
supplementary materials. It records how site-wise likelihood tables were
assembled, how the likelihood-deficit figure was generated, and how genes
enriched in highly opinionated sites were identified.

The original calculations were run in a separate server-side analysis
workspace. Set `ANALYSIS_ROOT` when adapting the table-construction or gene
mapping cells. Repository paths are used for the cleaned site-likelihood
tables and figure outputs.


In [ ]:
from pathlib import Path
import gzip
import os

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns


def find_repository_root(start=Path.cwd()):
    """Locate the repository from either its root or the Scripts directory."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Data").is_dir() and (candidate / "Scripts").is_dir():
            return candidate
    return start


REPO_ROOT = find_repository_root()
SITE_LH_DIR = REPO_ROOT / "Data" / "Analyses"
FIGURE_DIR = REPO_ROOT / "Figures"

# Root of the original server-side analysis workspace. This is only needed for
# the cells that reconstruct tables from the original IQ-TREE output.
ANALYSIS_ROOT = Path(
    os.environ.get("ANALYSIS_ROOT", "/path/to/original_analysis_workspace")
)
TREE_BUILDING_DIR = ANALYSIS_ROOT / "scratch" / "tree_building"
os.environ["ANALYSIS_ROOT"] = str(ANALYSIS_ROOT)

TOPOLOGIES = ["topoA1d", "topoA2d", "topoA3d", "topoA4d"]
MODELS = ["EDGEPROP", "SITEHET2"]
DATASETS = ["SOG10_tax_occ", "L38_synt", "L42_synt", "NP_synt"]
DATASET_MODELS = [
    (dataset, model)
    for dataset in DATASETS
    for model in MODELS
]

# The enrichment analysis reported in the supplement used only the three
# synteny-informed datasets.
ENRICHMENT_DATASET_MODELS = [
    (dataset, model)
    for dataset in ["L38_synt", "L42_synt", "NP_synt"]
    for model in MODELS
]

MODEL_LABELS = {
    "EDGEPROP": "{MAT}+I+G4",
    "SITEHET2": "LG+C60+F+G4",
}


## Prepare the site-wise likelihood tables


In [ ]:
%%bash
set -euo pipefail

: "${ANALYSIS_ROOT:?Set ANALYSIS_ROOT to the original analysis workspace}"
TREE_BUILDING_DIR="${ANALYSIS_ROOT}/scratch/tree_building"
TOPOLOGIES=(topoA1d topoA2d topoA3d topoA4d)
DATASETS=(SOG10_tax_occ L38_synt L42_synt NP_synt)
MODELS=(EDGEPROP SITEHET2)

for SUPERMATRIX in "${DATASETS[@]}"; do
  cd "${TREE_BUILDING_DIR}/${SUPERMATRIX}"
  for MODEL in "${MODELS[@]}"; do
    for TOPO in "${TOPOLOGIES[@]}"; do
      INPUT="${MODEL}.${SUPERMATRIX}.constrained_to_${TOPO}.sitelh"
      OUTPUT="${INPUT}.tsv"
      if [[ ! -s "$INPUT" ]]; then
        echo "Missing required input: ${PWD}/${INPUT}" >&2
        exit 1
      fi
      tail -n +2 "$INPUT" | tr ' ' '\n' > "$OUTPUT"
    done
  done
done


In [ ]:
SITE_LH_DIR.mkdir(parents=True, exist_ok=True)

for dataset, model in DATASET_MODELS:
    series = []
    for topology in TOPOLOGIES:
        source = (
            TREE_BUILDING_DIR
            / dataset
            / f"{model}.{dataset}.constrained_to_{topology}.sitelh.tsv"
        )
        if not source.is_file():
            raise FileNotFoundError(source)

        table = pd.read_csv(source, sep="\t", header=0)
        if table.shape[1] != 1:
            raise ValueError(f"Expected one likelihood column in {source}")
        series.append(table.iloc[:, 0].rename(topology))

    combined = pd.concat(series, axis=1)
    if list(combined.columns) != TOPOLOGIES:
        raise ValueError(f"Unexpected topology columns for {dataset}, {model}")

    destination = (
        SITE_LH_DIR
        / f"{model}.{dataset}.constrained_to_topo_d.sitelh.tsv.gz"
    )
    combined.to_csv(
        destination,
        sep="\t",
        index=True,
        compression={"method": "gzip", "compresslevel": 9, "mtime": 0},
    )
    print(f"Wrote {destination.name}: {combined.shape[0]:,} sites")


## Likelihood-deficit analysis for Topo A1–A4


In [ ]:
def plot_fourpanel(ax_grid, infile, topo_vector, ref_topo, palette, min_x=0, quantile_thr=0.99):
    """
    Render the four subplots (i–iv) into a provided 2x2 grid of axes.
    ax_grid should be [[axA, axB], [axC, axD]].
    """
    axA, axB = ax_grid[0]
    axC, axD = ax_grid[1]

    # Read once per panel
    df_all = pd.read_csv(infile, sep='\t', header=0, index_col=0)

    # X-axis padding
    x_pad = 0.02

    # ----------------------------
    # Panel A (top-left)
    # ----------------------------
    dat = df_all[topo_vector].copy().sort_values(ref_topo)
    sns.lineplot(
        data=dat,
        x=[x / dat.shape[0] for x in range(dat.shape[0])],
        y=ref_topo,
        ax=axA,
        color='black',
        legend=False
    )
    axA.set(xlabel='Cumulative distribution of sites', ylabel='Site-wise LL')
    axA.yaxis.set_minor_locator(ticker.AutoMinorLocator(3))
    axA.yaxis.set_minor_formatter(ticker.NullFormatter())
    axA.set_title('i', loc='left', y=0.85)
    axA.grid(True)

    # ----------------------------
    # Panel B (top-right)
    # ----------------------------
    dfB = df_all[topo_vector].copy()
    dfB['CV'] = 100 * np.abs(dfB[topo_vector].std(axis=1) / dfB[topo_vector].mean(axis=1))
    dfB['rel_CV'] = dfB.CV / dfB.CV.sum()
    sns.ecdfplot(data=dfB, y='rel_CV', complementary=False, ax=axB, color='black')
    axB.set_yscale('log')
    axB.set(xlabel='Cumulative distribution of sites', ylabel='Site-wise LL \ncoefficient of variation (%)')
    y_minor = ticker.LogLocator(base=10.0, subs=np.arange(1, 10.0) * 0.5, numticks=10)
    axB.yaxis.set_minor_locator(y_minor)
    axB.yaxis.set_minor_formatter(ticker.NullFormatter())
    axB.set_title('ii', loc='left', y=0.85)
    axB.grid(True)

    # ----------------------------
    # Panel C (bottom-left)
    # ----------------------------
    dfC = df_all[topo_vector].copy()
    dfC2 = dfC.sort_values(ref_topo, ascending=False).reset_index(drop=True)
    dfC2['ref_cum_LL'] = dfC2[ref_topo].cumsum() / dfC2[ref_topo].sum()
    stepC = max(1, len(dfC2) // 100)
    dat2C = dfC2.copy()
    dat2C[topo_vector] = dat2C[topo_vector].cumsum()
    dat2C = dat2C.iloc[::stepC, :]
    dat2C[topo_vector] = (1 - dat2C[topo_vector].div(dat2C[ref_topo], axis=0)) * 100
    dat2C = dat2C.reset_index(drop=True)
    num_points_C = len(dat2C)
    dat2C['frac'] = np.linspace(0, 1, num_points_C)
    dat2C_m = pd.melt(dat2C, id_vars=['frac', 'ref_cum_LL'], var_name='variable', value_name='value')

    sns.lineplot(
        data=dat2C_m,
        x='frac', y='value', hue='variable', palette=palette, ax=axC, legend=False
    )
    axC.set(
        xlabel="Site fraction\n sorted by decisiveness",
        ylabel=f"Cumulative LL\nrelative to {ref_topo[:-1]}\n(%)",
    )
    axC2 = axC.twinx()
    sns.lineplot(data=dat2C_m, x='frac', y='ref_cum_LL', ax=axC2, color='grey', legend=False)
    axC2.lines[0].set_linestyle("--")
    axC2.set(ylabel=f"{ref_topo} cumul. LL")
    axC2.yaxis.label.set_color('grey')
    axC2.tick_params(axis='y', colors='grey')
    axC.set_title('iii', loc='left', y=0.85)
    axC.grid(True)
    axC2.grid(False)

    # ----------------------------
    # Panel D (bottom-right)
    # ----------------------------
    dfD = df_all[topo_vector].copy()
    dfD['CV'] = 100 * np.abs(dfD[topo_vector].std(axis=1) / dfD[topo_vector].mean(axis=1))
    dfD2 = dfD.sort_values(['CV', ref_topo]).reset_index(drop=True)
    dfD2['ref_cum_LL'] = dfD2[ref_topo].cumsum() / dfD2[ref_topo].sum()
    stepD = max(1, len(dfD2) // 100)
    dat2D = dfD2.copy()
    dat2D[topo_vector] = dat2D[topo_vector].cumsum()
    dat2D = dat2D.iloc[::stepD, :]
    dat2D[topo_vector] = (1 - dat2D[topo_vector].div(dat2D[ref_topo], axis=0)) * 100
    dat2D = dat2D.reset_index(drop=True)
    num_points_D = len(dat2D)
    dat2D['frac'] = np.linspace(0, 1, num_points_D)
   
    dat2D_m = pd.melt(dat2D, id_vars=['frac', 'CV', 'ref_cum_LL'], var_name='variable', value_name='value')

    max_row = dat2D_m.value.idxmax()
    max_excess = dat2D_m.loc[max_row, 'value']
    second_top_topo = dat2D_m.loc[max_row, 'variable']
    frac_second = dat2D_m.loc[max_row, 'frac']
    slice=dat2D_m.loc[dat2D_m.frac==frac_second].sort_values(by='value',ascending=False,ignore_index=True)
    third_top_topo=slice.loc[1,'variable']
    frac_third=slice.loc[1,'value']
    
    print(f"{second_top_topo[:-1]} becomes the maximum-likelihood topology (LL = {ref_topo[:-1]} + {max_excess:.4f}%) ", f"when removing the top {(1-frac_second)*100:.0f}% of most opinionated sites.")
    print(f"the next best hypothesis is {third_top_topo[:-1]} with deltaL {frac_third:.4f}%")


    sns.lineplot(
        data=dat2D_m,
        x='frac', y='value', hue='variable', palette=palette, ax=axD, legend=True
    )
    axD.axvline(frac_second, 0, 1, color='r', linestyle=':', linewidth=1)
    axD.set(
        xlabel="Site fraction\n sorted by opinionatedness",
        ylabel=f"{ref_topo} cumul. LL deficit (%)",
    )
    axD2 = axD.twinx()
    sns.lineplot(data=dat2D_m, x='frac', y='ref_cum_LL', ax=axD2, color='grey', legend=False)
    axD2.lines[0].set_linestyle("--")
    axD2.set(ylabel=f"{ref_topo[:-1]}\ncumul. LL")
    axD2.yaxis.label.set_color('grey')
    axD2.tick_params(axis='y', colors='grey')
    axD.set_title('iv', loc='left', y=0.85)
    axD.grid(True)
    axD2.grid(False)
    sns.move_legend(axD, "upper left", bbox_to_anchor=(1.02, 1), ncol=1, title=None, frameon=False)

    # Share y between C and D
    axD.sharey(axC)

    # Harmonize y range using 99% range
    thr_min = (1 - quantile_thr) / 2
    thr_max = 1 - thr_min
    ymin = min(
        dat2C_m[dat2C_m.index > min_x]['value'].quantile(q=thr_min),
        dat2D_m[dat2D_m.index > min_x]['value'].quantile(q=thr_min)
    )
    ymax = max(
        dat2C_m[dat2C_m.index > min_x]['value'].quantile(q=thr_max),
        dat2D_m[dat2D_m.index > min_x]['value'].quantile(q=thr_max)
    )
    yrange = ymax - ymin if ymax > ymin else 1.0
    ymin -= 0.02 * yrange
    ymax += 0.02 * yrange
    axC.set_ylim(ymin, ymax)

    # X-limits and locators per column (apply to both top and bottom in each column)
    left_xlim = (min_x, 1 + x_pad)
    right_xlim = (min_x, 1 + x_pad)
    axC.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    axD.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    axC.set_xlim(left_xlim)
    axD.set_xlim(right_xlim)
    axA.set_xlim(left_xlim)
    axB.set_xlim(right_xlim)

    # Remove redundant labels on top row
    for ax in (axA, axB):
        ax.set_xlabel('')
        ax.tick_params(axis='x', labelbottom=False)

    # Move Y-axis of panel B to the right
    axB.yaxis.tick_right()
    axB.yaxis.set_label_position("right")

    # Hide twin y labels/ticks on C's right side
    axC2.set_ylabel('')
    axC2.tick_params(axis='y', labelright=False)

    # Hide y labels/ticks on D's left side
    axD.set_ylabel('')
    axD.tick_params(axis='y', labelleft=False)

sns.set_style("whitegrid")
plt.rcParams.update({"font.size": 8})

alpha = list("ABCDEFGH")
fig = plt.figure(figsize=(15, 12), constrained_layout=False)
gs_outer = fig.add_gridspec(nrows=4, ncols=2, hspace=0.35, wspace=0.25)

for panel_idx, (dataset, model) in enumerate(DATASET_MODELS):
    row = panel_idx // 2
    column = panel_idx % 2
    gs_inner = gs_outer[row, column].subgridspec(
        2, 2, height_ratios=[1, 3], hspace=0.12, wspace=0.15
    )
    axC = fig.add_subplot(gs_inner[1, 0])
    axD = fig.add_subplot(gs_inner[1, 1], sharex=axC)
    axA = fig.add_subplot(gs_inner[0, 0])
    axB = fig.add_subplot(gs_inner[0, 1])

    infile = (
        SITE_LH_DIR
        / f"{model}.{dataset}.constrained_to_topo_d.sitelh.tsv.gz"
    )
    df_all = pd.read_csv(infile, sep="\t", header=0, index_col=0)
    if list(df_all.columns) != TOPOLOGIES:
        raise ValueError(f"Unexpected topology columns in {infile}")

    ref_topo = df_all[TOPOLOGIES].sum().idxmax()
    palette = dict(
        zip(TOPOLOGIES, ["#b278f9", "#d972f9", "#b65f9f", "#ffb2fb"])
    )
    plot_fourpanel(
        [[axA, axB], [axC, axD]],
        infile,
        TOPOLOGIES,
        ref_topo,
        palette,
        min_x=0,
        quantile_thr=0.99,
    )

    axA.text(
        -0.12, 1.18, alpha[panel_idx],
        transform=axA.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left",
    )
    axA.text(
        0.0, 1.18,
        f"Dataset: {dataset}; model: {MODEL_LABELS[model]}",
        transform=axA.transAxes,
        fontsize=8,
        va="top",
        ha="left",
    )

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
figure_path = FIGURE_DIR / "topoAd_LLdeficit.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved {figure_path}")


## Remove the most opinionated sites from the alignments


In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord


def opinionated_site_cutoff(site_likelihoods, ref_topology):
    """Return the percentile at which an alternative reaches maximum LL excess."""
    df = site_likelihoods[TOPOLOGIES].copy()
    df["CV"] = 100 * np.abs(
        df[TOPOLOGIES].std(axis=1) / df[TOPOLOGIES].mean(axis=1)
    )
    ordered = df.sort_values(["CV", ref_topology]).reset_index(drop=True)
    cumulative = ordered.copy()
    cumulative[TOPOLOGIES] = cumulative[TOPOLOGIES].cumsum()
    step = max(1, len(cumulative) // 100)
    cumulative = cumulative.iloc[::step, :].copy()
    cumulative[TOPOLOGIES] = (
        1
        - cumulative[TOPOLOGIES].div(cumulative[ref_topology], axis=0)
    ) * 100
    cumulative["fraction"] = np.linspace(0, 1, len(cumulative))
    long = pd.melt(
        cumulative,
        id_vars=["fraction", "CV"],
        value_vars=TOPOLOGIES,
        var_name="topology",
        value_name="deltaL",
    )
    maximum = long.loc[long["deltaL"].idxmax()]
    return float(maximum["fraction"])


alignment_files = {
    "SOG10_tax_occ": (
        REPO_ROOT / "Data/Matrices/A144/SOG10_tax_occ.concatenated.fa.gz"
    ),
    "L38_synt": (
        REPO_ROOT / "Data/Matrices/Synteny_informed/L38_synt.concatenated.fa.gz"
    ),
    "L42_synt": (
        REPO_ROOT / "Data/Matrices/Synteny_informed/L42_synt.concatenated.fa.gz"
    ),
    "NP_synt": (
        REPO_ROOT / "Data/Matrices/Synteny_informed/NP_synt.concatenated.fa.gz"
    ),
}

for dataset, model in DATASET_MODELS:
    infile = (
        SITE_LH_DIR
        / f"{model}.{dataset}.constrained_to_topo_d.sitelh.tsv.gz"
    )
    site_likelihoods = pd.read_csv(
        infile, sep="\t", header=0, index_col=0
    )
    if list(site_likelihoods.columns) != TOPOLOGIES:
        raise ValueError(f"Unexpected topology columns in {infile}")

    ref_topology = site_likelihoods[TOPOLOGIES].sum().idxmax()
    cutoff = opinionated_site_cutoff(site_likelihoods, ref_topology)

    likelihoods = site_likelihoods[TOPOLOGIES].copy()
    likelihoods["CV"] = 100 * np.abs(
        likelihoods[TOPOLOGIES].std(axis=1)
        / likelihoods[TOPOLOGIES].mean(axis=1)
    )
    retained_sites = sorted(
        likelihoods.index[
            likelihoods["CV"] < likelihoods["CV"].quantile(cutoff)
        ].astype(int)
    )

    output = (
        TREE_BUILDING_DIR
        / dataset
        / f"without_most_opinionated_sites.{model}.{dataset}.concatenated.fa"
    )
    records = []
    with gzip.open(alignment_files[dataset], "rt") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            sequence = "".join(record.seq[position] for position in retained_sites)
            records.append(
                SeqRecord(Seq(sequence), id=record.id, description="")
            )
    SeqIO.write(records, output, "fasta")
    print(
        f"{dataset}, {model}: removed {(1 - cutoff) * 100:.1f}% of sites; "
        f"wrote {output.name}"
    )


## Enrichment test


In [ ]:
"""
Multi-dataset, fast mapping from concatenated alignment positions to genes.

- Registers each SUPERMATRIX with its partition (.partition.nex) and lengths TSV.
- Reuses a fast, vectorized index per dataset (no Python range objects).
- O(m log n) lookup using numpy.searchsorted for m positions and n genes.
- Preserves filename extensions from the partition file.

APIs:
- register_dataset(SUPERMATRIX, partition_path, lengths_tsv, file_col=0, length_col=2, sep="\t")
- map_positions(SUPERMATRIX, positions, pos_base=0, return_offsets=True, gene_pos_base=0) -> DataFrame
- lookup_single_position(SUPERMATRIX, position, pos_base=0, gene_pos_base=0) -> (gene, gene_index, gene_pos)
- get_len_dict(SUPERMATRIX) -> dict(file -> length)

Notes:
- positions: pos_base=0 means positions are 0-based; use pos_base=1 if they are 1-based.
- gene_pos_base controls the base of the within-gene position (0- or 1-based).
"""

import os
import re
import numpy as np
import pandas as pd


# ------------------------------
# Parsing and index construction
# ------------------------------

def parse_partition_nex(partition_path):
    """
    Parse a .partition.nex file to extract the ordered list of gene files (OG_list),
    preserving the original filename including its extension (e.g., '.fa', '.faa').

    Returns:
        og_list (list[str]): ordered list of filenames with extensions, as they appear
                             in the partition file (basename only, path stripped).
    """
    OG_list=[]
    with open(partition_path,'r') as f:
        lines=f.read().splitlines()
        i=2
        while i < len(lines):
            if lines[i][:12]=='    charset ':
                # print(lines[i])
                OG_list+=[lines[i].split('/')[-1][:-4]]
                i+=1
            else:
                break
    return(OG_list)


def load_gene_lengths(lengths_tsv, file_col=0, length_col=2, sep="\t"):
    """
    Load the table of all genes and their lengths.

    Parameters:
        lengths_tsv: path to TSV with at least 'file' and 'length' columns.
                     By default, uses column 0 as 'file' and column 2 as 'len'.
        file_col: integer or column name for file column
        length_col: integer or column name for length column
        sep: delimiter

    Returns:
        DataFrame with columns: file (str), len (int)
    """
    df = pd.read_csv(lengths_tsv, sep=sep, header=None)
    df = df.iloc[:, [file_col, length_col]].copy()
    df.columns = ["file", "len"]
    df["file"] = df["file"].astype(str)
    df["len"] = pd.to_numeric(df["len"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["len"]).astype({"len": np.int64})
    return df


def build_gene_index(og_list, lengths_df):
    """
    Build a fast index for mapping positions to genes.

    Uses 0-based, end-exclusive cumulative boundaries (ends_exclusive).
    For gene i with length L_i, the coverage is:
        [start_i, end_i) = [sum_{j<i} L_j, sum_{j<=i} L_j)

    Returns:
        index_dict with:
            files: np.ndarray[str], shape (n,)
            lens: np.ndarray[int], shape (n,)
            ends_exclusive: np.ndarray[int], shape (n,) cumulative end per gene
            starts_0b: np.ndarray[int], shape (n,) starting positions (0-based)
            total_length: int
            len_dict: dict(file -> length)
    """
    # Reindex lengths to the order found in the partition file (og_list)
    lengths_df = lengths_df.set_index("file").reindex(og_list)
    # If any genes are missing lengths, raise an error
    if lengths_df["len"].isnull().any():
        missing = lengths_df[lengths_df["len"].isnull()].index.tolist()
        raise ValueError(
            f"Missing lengths for {len(missing)} genes found in partition but not in length table. "
            f"Examples: {missing[:5]}"
        )

    lens = lengths_df["len"].to_numpy(dtype=np.int64)
    files = lengths_df.index.to_numpy(dtype=object)

    ends_exclusive = np.cumsum(lens)  # end boundary (exclusive) for each gene
    starts_0b = ends_exclusive - lens  # start (0-based inclusive)

    index = {
        "files": files,
        "lens": lens,
        "ends_exclusive": ends_exclusive,
        "starts_0b": starts_0b,
        "total_length": int(ends_exclusive[-1]),
        "len_dict": dict(zip(files.tolist(), lens.tolist())),
    }
    return index


# ------------------------------
# Global registries for multiple datasets
# ------------------------------

_INDEX_REGISTRY = {}
_LEN_REGISTRY = {}
_META = {}  # optional: store paths, etc.


def register_dataset(
    supermatrix,
    partition_path,
    lengths_tsv,
    file_col=0,
    length_col=2,
    sep="\t",
):
    """
    Register a dataset by SUPERMATRIX name.

    Builds and caches the gene index so repeated queries are fast.

    Parameters:
        supermatrix: dataset key/name (str)
        partition_path: path to {SUPERMATRIX}.partition.nex
        lengths_tsv: path to lengths TSV for this dataset
        file_col, length_col, sep: passed to load_gene_lengths
    """
    og_list = parse_partition_nex(partition_path)
    lengths_df = load_gene_lengths(lengths_tsv, file_col=file_col, length_col=length_col, sep=sep)
    index = build_gene_index(og_list, lengths_df)

    _INDEX_REGISTRY[supermatrix] = index
    _LEN_REGISTRY[supermatrix] = index["len_dict"]
    _META[supermatrix] = {
        "partition_path": partition_path,
        "lengths_tsv": lengths_tsv,
        "n_genes": len(og_list),
        "total_length": index["total_length"],
    }
    return index


def get_len_dict(supermatrix):
    if supermatrix not in _LEN_REGISTRY:
        raise KeyError(f"SUPERMATRIX '{supermatrix}' is not registered. Call register_dataset first.")
    return _LEN_REGISTRY[supermatrix]


# ------------------------------
# Fast mapping functions
# ------------------------------

def _ensure_registered(supermatrix):
    if supermatrix not in _INDEX_REGISTRY:
        raise KeyError(f"SUPERMATRIX '{supermatrix}' is not registered. Call register_dataset first.")
    return _INDEX_REGISTRY[supermatrix]


def map_positions(supermatrix, positions, pos_base=0, return_offsets=True, gene_pos_base=0):
    """
    Map many positions to genes for a registered SUPERMATRIX.

    Parameters:
        supermatrix: dataset key/name (str)
        positions: array-like of positions (int). Can be list, numpy array, or pandas Series.
        pos_base: 0 if positions are 0-based (first column is 0), 1 if 1-based
        return_offsets: if True, also return position within gene
        gene_pos_base: 0 or 1, base for gene-local positions

    Returns:
        DataFrame with columns:
            - position
            - gene (file name)
            - gene_index (0-based index of gene)
            - gene_pos (if return_offsets=True; position within gene, with selected base)
    """
    index = _ensure_registered(supermatrix)

    positions = np.asarray(positions, dtype=np.int64)
    if positions.ndim != 1:
        positions = positions.ravel()

    # Normalize to 0-based coordinates
    pos0 = positions - (1 if pos_base == 1 else 0)

    ends = index["ends_exclusive"]
    starts = index["starts_0b"]
    files = index["files"]
    total_len = index["total_length"]

    # Out-of-bounds mask
    oob = (pos0 < 0) | (pos0 >= total_len)

    # i = index of the first cumulative end > pos (per position)
    i = np.searchsorted(ends, pos0, side="right")

    # Build outputs
    gene = np.full_like(pos0, fill_value=object(), dtype=object)
    gene_index = np.full(pos0.shape, -1, dtype=np.int64)

    # Valid mask
    valid = ~oob
    if np.any(valid):
        iv = i[valid]
        gene_index[valid] = iv
        gene[valid] = files[iv]

    out = {
        "position": positions,
        "gene": gene,
        "gene_index": gene_index,
    }

    if return_offsets:
        # For valid entries, compute position within gene: pos0 - start + base
        gene_pos = np.full(pos0.shape, -1, dtype=np.int64)
        if np.any(valid):
            gene_pos[valid] = pos0[valid] - starts[i[valid]] + (1 if gene_pos_base == 1 else 0)
        out["gene_pos"] = gene_pos

    # Mark OOB clearly
    if np.any(oob):
        out["gene"][oob] = None
        out["gene_index"][oob] = -1
        if return_offsets:
            out["gene_pos"][oob] = -1

    return pd.DataFrame(out)


def lookup_single_position(supermatrix, position, pos_base=0, gene_pos_base=0):
    """
    Fast single-position lookup for a registered SUPERMATRIX.
    Returns (gene_name or None, gene_index, gene_pos_within_gene or -1 if OOB).
    """
    index = _ensure_registered(supermatrix)

    pos0 = int(position) - (1 if pos_base == 1 else 0)
    ends = index["ends_exclusive"]
    starts = index["starts_0b"]
    files = index["files"]
    total_len = index["total_length"]

    if pos0 < 0 or pos0 >= total_len:
        return None, -1, -1

    i = int(np.searchsorted(ends, pos0, side="right"))
    gene_name = files[i]
    gene_pos = pos0 - starts[i] + (1 if gene_pos_base == 1 else 0)
    return gene_name, i, gene_pos

# Register the three synteny-informed datasets used in the enrichment test.
# These paths describe the original analysis workspace.
register_dataset(
    "L42_synt",
    ANALYSIS_ROOT / "scratch/tree_building/L42_synt/L42_synt.partition.nex",
    ANALYSIS_ROOT / "scratch/tree_building/L42_synteny/phykit/pk_aln_len_no_gaps.tsv",
    file_col=0,
    length_col=2,
    sep="\t",
)
register_dataset(
    "L38_synt",
    ANALYSIS_ROOT / "scratch/tree_building/L38_synt/L38_synt.partition.nex",
    ANALYSIS_ROOT / "scratch/tree_building/L42_synteny/phykit/pk_aln_len_no_gaps.tsv",
    file_col=0,
    length_col=2,
    sep="\t",
)
register_dataset(
    "NP_synt",
    ANALYSIS_ROOT / "scratch/tree_building/NP_synt/NP_synt.partition.nex",
    ANALYSIS_ROOT / "data/gene_alignments/early_branching/phykit/pk_aln_len_no_gaps.tsv",
    file_col=0,
    length_col=2,
    sep="\t",
)


In [ ]:
def count_genes_enriched_for_ML_topology(dat):

    import pandas as pd
    import numpy as np
    from scipy.stats import poisson, binom
    from statsmodels.stats.multitest import multipletests
    
    # Validate and clean input
    required_cols = {"file", "flag_count", "gene_len"}
    missing = required_cols - set(dat.columns)
    if missing:
        raise ValueError(f"Input DataFrame 'dat' is missing required columns: {missing}")
    
    # Work on a clean copy and enforce types
    df = dat[["file", "flag_count", "gene_len"]].copy()
    df["flag_count"] = pd.to_numeric(df["flag_count"], errors="coerce").fillna(0).astype(int)
    df["gene_len"] = pd.to_numeric(df["gene_len"], errors="coerce").astype(int)
    
    # Drop rows with non-positive lengths (cannot define per-base rate)
    df = df[df["gene_len"] > 0].reset_index(drop=True)
    if df.empty:
        raise ValueError("No rows with positive gene_len found.")
    
    # Background totals (union of genes as background)
    G = int(df["gene_len"].sum())        # total background length (bp)
    K = int(df["flag_count"].sum())      # total flagged within background
    if G <= 0:
        raise ValueError("Total background length must be positive.")
    
    # Global per-base rate / probability
    r = K / G                            # rate per bp for Poisson
    p = r                                # per-base probability for Binomial (same value numerically)
    
    # Per-gene expected flags and p-values (Poisson test)
    mu = r * df["gene_len"].values
    x = df["flag_count"].values
    
    pvals_poisson = poisson.sf(x - 1, mu)             # one-sided P[X >= x | mu]
    _, qvals_poisson, _, _ = multipletests(pvals_poisson, method="fdr_bh")
    
    # Enrichment metrics
    with np.errstate(divide="ignore", invalid="ignore"):
        enrichment_ratio = np.where(mu > 0, x / mu, np.nan)
        log2_enrichment = np.log2(enrichment_ratio)
    
    # Assemble Poisson results
    results_poisson = df.copy()
    results_poisson["expected_flags"] = mu
    results_poisson["rate_per_bp"] = r
    results_poisson["p_poisson"] = pvals_poisson
    results_poisson["q_fdr_bh"] = qvals_poisson
    results_poisson["enrichment_ratio"] = enrichment_ratio
    results_poisson["log2_enrichment"] = log2_enrichment
    
    # Sort by significance and effect size
    results_poisson = results_poisson.sort_values(
        ["q_fdr_bh", "p_poisson", "log2_enrichment"],
        ascending=[True, True, False]
    ).reset_index(drop=True)
    
    # Attach summary stats as attributes (optional)
    results_poisson.attrs["background_length_bp"] = G
    results_poisson.attrs["total_flagged"] = K
    results_poisson.attrs["global_rate_per_bp"] = r
    
    # Optional: Binomial test (similar to Poisson, useful as a cross-check)
    n = df["gene_len"].values
    pvals_binom = binom.sf(x - 1, n, p)
    _, qvals_binom, _, _ = multipletests(pvals_binom, method="fdr_bh")
    
    results_binom = df[["file"]].copy()
    results_binom["p_binom"] = pvals_binom
    results_binom["q_fdr_bh_binom"] = qvals_binom
    results_binom["per_base_prob"] = p
    
    # Merge Poisson and Binomial results for convenience
    results_merged = results_poisson.merge(results_binom, on="file")
    return(results_merged[(results_merged.q_fdr_bh_binom<0.05) & (results_merged.q_fdr_bh<0.05)], results_merged)

all_enrichment_results = []
for dataset, model in ENRICHMENT_DATASET_MODELS:
    infile = (
        SITE_LH_DIR
        / f"{model}.{dataset}.constrained_to_topo_d.sitelh.tsv.gz"
    )
    site_likelihoods = pd.read_csv(
        infile, sep="\t", header=0, index_col=0
    )
    if list(site_likelihoods.columns) != TOPOLOGIES:
        raise ValueError(f"Unexpected topology columns in {infile}")

    ref_topology = site_likelihoods[TOPOLOGIES].sum().idxmax()
    cutoff = opinionated_site_cutoff(site_likelihoods, ref_topology)

    likelihoods = site_likelihoods[TOPOLOGIES].copy()
    likelihoods["CV"] = 100 * np.abs(
        likelihoods[TOPOLOGIES].std(axis=1)
        / likelihoods[TOPOLOGIES].mean(axis=1)
    )
    flagged_sites = likelihoods.index[
        likelihoods["CV"] >= likelihoods["CV"].quantile(cutoff)
    ].to_numpy()

    mapped = map_positions(dataset, flagged_sites)
    if mapped["gene"].isna().any():
        raise ValueError(f"Some {dataset} sites could not be mapped to genes")

    gene_counts = (
        mapped["gene"]
        .value_counts()
        .rename_axis("file")
        .reset_index(name="flag_count")
    )
    gene_counts["gene_len"] = gene_counts["file"].map(get_len_dict(dataset))

    significant, complete = count_genes_enriched_for_ML_topology(gene_counts)
    complete["dataset"] = dataset
    complete["model"] = model
    all_enrichment_results.append(complete)

    print(
        f"{dataset}, {model}: {len(flagged_sites):,} flagged sites; "
        f"{significant.shape[0]:,} significantly enriched genes"
    )

enrichment_results = pd.concat(all_enrichment_results, ignore_index=True)
enrichment_path = SITE_LH_DIR / "LLdeficit_topoAd_enrichment.tsv"
enrichment_results.to_csv(enrichment_path, sep="\t", index=False)
print(f"Wrote {enrichment_path}")
